# V11 Final Control Experiments: Process vs Sampling Artifact

This notebook runs the final focused controls for the Q1 manuscript.

Goal: test whether the strong noise + imbalance interaction is mainly a process/sampling-frame effect.

It runs only the essential controls:

1. **Original process**: label noise → imbalance using corrupted observed labels  
2. **Clean-label sampling control / reversed process**: imbalance using clean labels → label noise  
3. **Fixed current-index control**: original compound retained rows are reused for the matched imbalance baseline  
4. **Equal-training-size control**: clean, single, and compound conditions are compared after matching the retained training size

Default setting:

- 24 PMLB datasets
- 9 models
- 5 seeds
- noise = 0.2
- imbalance = 0.8
- missingness = 0.0

Final output:

`/kaggle/working/v11_process_control_outputs.zip`

Recommended Kaggle settings:

- Internet: ON
- Accelerator: None


In [ ]:
# =========================
# CONFIG
# =========================

OUT_ROOT = "/kaggle/working/v11_process_control"

PMLB_DATASETS = [
    "adult", "mushroom", "allhypo", "clean2", "dna", "hypothyroid", "kr_vs_kp",
    "magic", "nursery", "optdigits", "pendigits", "phoneme", "satimage",
    "segmentation", "spambase", "splice", "tic_tac_toe", "vehicle",
    "vowel", "waveform_40", "mfeat_fourier", "mfeat_karhunen",
    "mfeat_morphological", "mfeat_zernike"
]

MODELS = [
    "LogisticRegression",
    "LinearSVM",
    "KNN",
    "DecisionTree",
    "RandomForest",
    "ExtraTrees",
    "HistGradientBoosting",
    "XGBoost",
    "LightGBM",
]

SEEDS = [0, 1, 2, 3, 4]
MAX_ROWS_PER_DATASET = 20000

NOISE_RATE = 0.20
IMBALANCE_RATIO = 0.80
TEST_SIZE = 0.25

# Resume support: if True, existing rows are skipped.
RESUME = True

# Keep False for final run. Set True only for smoke testing.
SMOKE_TEST = False
if SMOKE_TEST:
    PMLB_DATASETS = PMLB_DATASETS[:3]
    MODELS = ["LogisticRegression", "RandomForest", "XGBoost"]
    SEEDS = [0]

print("Datasets:", len(PMLB_DATASETS))
print("Models:", len(MODELS))
print("Seeds:", SEEDS)
print("Noise:", NOISE_RATE, "Imbalance:", IMBALANCE_RATIO)


In [ ]:
# =========================
# INSTALLS + IMPORTS
# =========================

import os, sys, subprocess, warnings, time, json, glob, shutil
from pathlib import Path
from typing import Dict, Tuple, List, Any

warnings.filterwarnings("ignore")
os.environ["PYTHONWARNINGS"] = "ignore"

for pkg in ["pmlb", "xgboost", "lightgbm"]:
    try:
        if pkg == "pmlb":
            import pmlb
        elif pkg == "xgboost":
            import xgboost
        elif pkg == "lightgbm":
            import lightgbm
    except Exception:
        print("Installing", pkg)
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])

import numpy as np
import pandas as pd

from pmlb import fetch_data

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.metrics import accuracy_score, balanced_accuracy_score, f1_score, matthews_corrcoef, recall_score

from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier, HistGradientBoostingClassifier

from xgboost import XGBClassifier
from lightgbm import LGBMClassifier

Path(OUT_ROOT).mkdir(parents=True, exist_ok=True)
print("Setup complete")


In [ ]:
# =========================
# UTILITIES
# =========================


def ensure_dir(path):
    p = Path(path)
    p.mkdir(parents=True, exist_ok=True)
    return p


def save_row(row, path):
    path = Path(path)
    pd.DataFrame([row]).to_csv(path, mode="a", header=not path.exists(), index=False)


def clean_columns(df):
    df = df.copy()
    df.columns = [str(c).strip().replace(" ", "_").replace("/", "_") for c in df.columns]
    return df


def encode_target(y):
    le = LabelEncoder()
    return pd.Series(le.fit_transform(pd.Series(y).astype(str))).reset_index(drop=True)


def stratified_cap(X, y, max_rows=20000, seed=123):
    X = X.reset_index(drop=True)
    y = pd.Series(y).reset_index(drop=True)
    if len(X) <= max_rows:
        return X, y, False
    try:
        Xs, _, ys, _ = train_test_split(X, y, train_size=max_rows, random_state=seed, stratify=y)
        return Xs.reset_index(drop=True), ys.reset_index(drop=True), True
    except Exception:
        rng = np.random.default_rng(seed)
        idx = rng.choice(np.arange(len(X)), size=max_rows, replace=False)
        return X.iloc[idx].reset_index(drop=True), y.iloc[idx].reset_index(drop=True), True


def load_pmlb_dataset(name):
    df = fetch_data(name, local_cache_dir="/kaggle/working/pmlb_cache")
    df = clean_columns(df)
    target_col = "target" if "target" in df.columns else df.columns[-1]
    X = df.drop(columns=[target_col])
    y = encode_target(df[target_col])
    keep = [c for c in X.columns if X[c].nunique(dropna=True) > 1]
    X = X[keep].copy()
    X, y, capped = stratified_cap(X, y, max_rows=MAX_ROWS_PER_DATASET, seed=123)
    meta = {
        "dataset_name": name,
        "n_rows": int(len(X)),
        "n_features": int(X.shape[1]),
        "n_classes": int(y.nunique()),
        "majority_ratio": float(y.value_counts(normalize=True).max()),
        "minority_count": int(y.value_counts().min()),
        "capped_20000": bool(capped),
    }
    return X, y, meta


def validate_dataset(X, y, min_rows=100, min_minority=5):
    if len(X) < min_rows:
        return False, "too_few_rows"
    if pd.Series(y).nunique() < 2:
        return False, "too_few_classes"
    if pd.Series(y).value_counts().min() < min_minority:
        return False, "minority_too_small"
    return True, "valid"


def save_manifest():
    out_dir = ensure_dir(Path(OUT_ROOT) / "data_manifest")
    rows, failed = [], []
    for name in PMLB_DATASETS:
        try:
            X, y, meta = load_pmlb_dataset(name)
            valid, reason = validate_dataset(X, y)
            meta.update({"valid": valid, "reason": reason})
            rows.append(meta)
            print("Loaded", name, X.shape, "classes", y.nunique())
        except Exception as e:
            failed.append({"dataset_name": name, "error": str(e)[:500]})
            print("Failed", name, e)
    inc = pd.DataFrame(rows)
    fail = pd.DataFrame(failed)
    inc.to_csv(out_dir / "pmlb_manifest_v11_process_controls.csv", index=False)
    fail.to_csv(out_dir / "pmlb_manifest_failed_v11.csv", index=False)
    display(inc)
    return inc, fail


In [ ]:
# =========================
# CORRUPTION / SAMPLING CONTROLS
# =========================


def apply_label_noise_to_y(y, rate, seed):
    rng = np.random.default_rng(seed)
    y = pd.Series(y).reset_index(drop=True)
    arr = y.to_numpy().copy()
    flip_mask = np.zeros(len(arr), dtype=bool)
    if rate <= 0:
        return pd.Series(arr), flip_mask
    classes = np.unique(arr)
    n_flip = int(round(rate * len(arr)))
    if n_flip <= 0:
        return pd.Series(arr), flip_mask
    flip_idx = rng.choice(np.arange(len(arr)), size=n_flip, replace=False)
    for i in flip_idx:
        choices = classes[classes != arr[i]]
        if len(choices) > 0:
            arr[i] = rng.choice(choices)
            flip_mask[i] = True
    return pd.Series(arr), flip_mask


def induce_imbalance_indices(y_for_sampling, target_majority_ratio, seed):
    """Return retained indices after inducing imbalance from the supplied sampling labels.

    The majority class is recomputed from y_for_sampling. All majority examples are kept.
    Observed non-majority examples are downsampled so that majority share is approximately
    target_majority_ratio. In multiclass data, at least one example from each non-majority
    class is preserved when feasible.
    """
    y = pd.Series(y_for_sampling).reset_index(drop=True)
    rng = np.random.default_rng(seed)
    counts = y.value_counts()
    majority_class = counts.idxmax()
    majority_idx = y[y == majority_class].index.to_numpy()
    nonmajority_idx_all = y[y != majority_class].index.to_numpy()
    nonmajority_classes = [c for c in counts.index if c != majority_class]

    n_majority = len(majority_idx)
    desired_total = int(round(n_majority / float(target_majority_ratio)))
    desired_nonmajority = max(desired_total - n_majority, len(nonmajority_classes))
    desired_nonmajority = min(desired_nonmajority, len(nonmajority_idx_all))

    selected = []
    budget = desired_nonmajority
    for c in nonmajority_classes:
        cidx = y[y == c].index.to_numpy()
        if len(cidx) > 0 and budget > 0:
            selected.append(int(rng.choice(cidx)))
            budget -= 1
    if budget > 0:
        used = set(selected)
        available = np.array([i for i in nonmajority_idx_all if i not in used], dtype=int)
        if len(available) > 0:
            extra = rng.choice(available, size=min(budget, len(available)), replace=False)
            selected.extend([int(i) for i in extra])

    keep_idx = np.concatenate([majority_idx, np.array(selected, dtype=int)])
    rng.shuffle(keep_idx)

    info = {
        "sampling_majority_class": int(majority_class),
        "desired_nonmajority": int(desired_nonmajority),
        "actual_nonmajority": int(len(selected)),
        "n_retained": int(len(keep_idx)),
        "majority_ratio_after_sampling": float(y.iloc[keep_idx].value_counts(normalize=True).max()),
        "min_class_count_after_sampling": int(y.iloc[keep_idx].value_counts().min()),
    }
    return keep_idx.astype(int), info


def random_stratified_or_uniform_indices(y, n, seed):
    y = pd.Series(y).reset_index(drop=True)
    n = int(min(n, len(y)))
    if n <= 0:
        return np.array([], dtype=int)
    try:
        idx_all = np.arange(len(y))
        idx, _ = train_test_split(idx_all, train_size=n, random_state=seed, stratify=y)
        return np.array(idx, dtype=int)
    except Exception:
        rng = np.random.default_rng(seed)
        return rng.choice(np.arange(len(y)), size=n, replace=False).astype(int)


def downsample_given_indices(indices, n, seed):
    indices = np.asarray(indices, dtype=int)
    n = int(min(n, len(indices)))
    if len(indices) == n:
        return indices
    rng = np.random.default_rng(seed)
    return rng.choice(indices, size=n, replace=False).astype(int)


def composition_diag(y_clean, y_train_used, retained_idx, flip_mask=None):
    y_clean = pd.Series(y_clean).reset_index(drop=True)
    retained_idx = np.asarray(retained_idx, dtype=int)
    true_retained = y_clean.iloc[retained_idx].reset_index(drop=True)
    counts = y_clean.value_counts()
    clean_minority = counts.idxmin()
    clean_majority = counts.idxmax()
    out = {
        "train_n": int(len(retained_idx)),
        "clean_minority_class": int(clean_minority),
        "clean_majority_class": int(clean_majority),
        "clean_minority_retained_count": int((true_retained == clean_minority).sum()),
        "clean_minority_retention_rate": float((true_retained == clean_minority).sum() / max((y_clean == clean_minority).sum(), 1)),
        "effective_clean_minority_ratio": float((true_retained == clean_minority).sum() / max(len(retained_idx), 1)),
        "clean_true_majority_ratio_retained": float(true_retained.value_counts(normalize=True).max()),
        "clean_true_min_class_count_retained": int(true_retained.value_counts().min()),
    }
    if flip_mask is not None:
        fm = np.asarray(flip_mask)[retained_idx]
        out.update({
            "mislabeled_retained_count": int(fm.sum()),
            "mislabeled_retained_ratio": float(fm.sum() / max(len(retained_idx), 1)),
            "mislabeled_clean_minority_count": int(((true_retained == clean_minority).to_numpy() & fm).sum()),
            "mislabeled_clean_majority_count": int(((true_retained == clean_majority).to_numpy() & fm).sum()),
        })
    else:
        out.update({
            "mislabeled_retained_count": 0,
            "mislabeled_retained_ratio": 0.0,
            "mislabeled_clean_minority_count": 0,
            "mislabeled_clean_majority_count": 0,
        })
    return out


In [ ]:
# =========================
# MODEL DEFINITIONS
# =========================


def detect_cols(X):
    numeric = [c for c in X.columns if pd.api.types.is_numeric_dtype(X[c])]
    categorical = [c for c in X.columns if c not in numeric]
    return numeric, categorical


def make_preprocessor(X, scale=False):
    numeric, categorical = detect_cols(X)
    if scale:
        num_pipe = Pipeline([("impute", SimpleImputer(strategy="mean")), ("scale", StandardScaler())])
    else:
        num_pipe = Pipeline([("impute", SimpleImputer(strategy="mean"))])
    try:
        ohe = OneHotEncoder(handle_unknown="ignore", sparse_output=False)
    except TypeError:
        ohe = OneHotEncoder(handle_unknown="ignore", sparse=False)
    cat_pipe = Pipeline([("impute", SimpleImputer(strategy="most_frequent")), ("onehot", ohe)])
    transformers = []
    if numeric:
        transformers.append(("num", num_pipe, numeric))
    if categorical:
        transformers.append(("cat", cat_pipe, categorical))
    return ColumnTransformer(transformers, remainder="drop")


def get_model(model_name, X, seed, n_classes):
    scale = model_name in ["LogisticRegression", "LinearSVM", "KNN"]
    pre = make_preprocessor(X, scale=scale)

    if model_name == "LogisticRegression":
        clf = LogisticRegression(max_iter=2000, random_state=seed, n_jobs=-1)
    elif model_name == "LinearSVM":
        clf = LinearSVC(random_state=seed, max_iter=5000)
    elif model_name == "KNN":
        clf = KNeighborsClassifier(n_neighbors=5)
    elif model_name == "DecisionTree":
        clf = DecisionTreeClassifier(random_state=seed)
    elif model_name == "RandomForest":
        clf = RandomForestClassifier(n_estimators=200, random_state=seed, n_jobs=-1)
    elif model_name == "ExtraTrees":
        clf = ExtraTreesClassifier(n_estimators=200, random_state=seed, n_jobs=-1)
    elif model_name == "HistGradientBoosting":
        clf = HistGradientBoostingClassifier(random_state=seed, max_iter=200)
    elif model_name == "XGBoost":
        if n_classes == 2:
            clf = XGBClassifier(
                objective="binary:logistic", eval_metric="logloss",
                n_estimators=220, learning_rate=0.05, max_depth=6,
                subsample=0.9, colsample_bytree=0.9,
                tree_method="hist", random_state=seed, n_jobs=-1
            )
        else:
            clf = XGBClassifier(
                objective="multi:softprob", eval_metric="mlogloss", num_class=n_classes,
                n_estimators=220, learning_rate=0.05, max_depth=6,
                subsample=0.9, colsample_bytree=0.9,
                tree_method="hist", random_state=seed, n_jobs=-1
            )
    elif model_name == "LightGBM":
        clf = LGBMClassifier(
            objective="binary" if n_classes == 2 else "multiclass",
            n_estimators=220, learning_rate=0.05, num_leaves=31,
            subsample=0.9, colsample_bytree=0.9,
            random_state=seed, n_jobs=-1, verbose=-1
        )
    else:
        raise ValueError(model_name)

    return Pipeline([("pre", pre), ("clf", clf)])


def compute_metrics(model, X_test, y_test):
    y_pred = model.predict(X_test)
    y_test = pd.Series(y_test).reset_index(drop=True)
    labels = np.unique(y_test)
    out = {
        "accuracy": accuracy_score(y_test, y_pred),
        "balanced_accuracy": balanced_accuracy_score(y_test, y_pred),
        "macro_f1": f1_score(y_test, y_pred, average="macro", zero_division=0),
        "weighted_f1": f1_score(y_test, y_pred, average="weighted", zero_division=0),
        "mcc": matthews_corrcoef(y_test, y_pred),
    }
    try:
        minority_class = y_test.value_counts().idxmin()
        recalls = recall_score(y_test, y_pred, average=None, labels=labels, zero_division=0)
        out["minority_recall"] = float(recalls[list(labels).index(minority_class)])
    except Exception:
        out["minority_recall"] = np.nan
    return {k: float(v) for k, v in out.items()}


def fit_eval_condition(dataset_name, model_name, seed, condition_name, Xtr, ytr_used, Xte, yte, retained_idx, extra_diag=None):
    n_classes = int(pd.Series(yte).nunique())
    model = get_model(model_name, Xtr, seed, n_classes)
    t0 = time.time()
    model.fit(Xtr, ytr_used)
    metrics = compute_metrics(model, Xte, yte)
    row = {
        "dataset_name": dataset_name,
        "model_name": model_name,
        "seed": seed,
        "condition_name": condition_name,
        "train_n": int(len(ytr_used)),
        "n_classes_train": int(pd.Series(ytr_used).nunique()),
        "fit_eval_seconds": round(time.time() - t0, 4),
    }
    row.update(metrics)
    if extra_diag:
        row.update(extra_diag)
    return row


In [ ]:
# =========================
# BUILD CONDITIONS FOR ONE DATASET/SEED
# =========================


def make_conditions_for_seed(Xtr, ytr, seed):
    """Return a dict of train conditions and diagnostics for process-control tests."""
    Xtr = Xtr.reset_index(drop=True)
    ytr = pd.Series(ytr).reset_index(drop=True)

    # Full clean and full noise-only.
    full_idx = np.arange(len(Xtr), dtype=int)
    y_noise_full, flip_full = apply_label_noise_to_y(ytr, NOISE_RATE, seed + 1001)

    # Imbalance-only using clean labels.
    idx_imb_clean, info_imb_clean = induce_imbalance_indices(ytr, IMBALANCE_RATIO, seed + 2001)

    # Original compound: noise first, then imbalance using corrupted observed labels.
    idx_current, info_current = induce_imbalance_indices(y_noise_full, IMBALANCE_RATIO, seed + 3001)

    # Clean-label sampling/reversed: use clean imbalance indices, then corrupt labels on retained rows.
    y_noise_after_clean_imb_full, flip_after_clean_imb_full = apply_label_noise_to_y(ytr, NOISE_RATE, seed + 4001)

    # Fixed current-index control: same rows as original compound but clean labels for matched imbalance baseline.
    # The compound labels are the labels from the full noisy vector used to define current observed sampling.

    # Equal-size control: match all conditions to min size across clean-imbalance and current-compound samples.
    n_equal = int(min(len(idx_current), len(idx_imb_clean), len(Xtr)))
    idx_clean_equal = random_stratified_or_uniform_indices(ytr, n_equal, seed + 5001)
    idx_noise_equal = idx_clean_equal.copy()
    idx_imb_equal = downsample_given_indices(idx_imb_clean, n_equal, seed + 5002)
    idx_current_equal = downsample_given_indices(idx_current, n_equal, seed + 5003)
    y_noise_equal_full, flip_equal_full = apply_label_noise_to_y(ytr, NOISE_RATE, seed + 5004)

    conditions = {}

    def add(name, idx, y_vector, flip_mask=None, process_family="shared", sampling_info=None):
        idx = np.asarray(idx, dtype=int)
        X_used = Xtr.iloc[idx].reset_index(drop=True)
        y_used = pd.Series(y_vector).iloc[idx].reset_index(drop=True)
        diag = composition_diag(ytr, y_used, idx, flip_mask=flip_mask)
        diag.update({
            "process_family": process_family,
            "retained_index_count": int(len(idx)),
            "noise_rate": NOISE_RATE,
            "imbalance_ratio": IMBALANCE_RATIO,
        })
        if sampling_info:
            for k, v in sampling_info.items():
                diag[f"sampling_{k}"] = v
        conditions[name] = {"X": X_used, "y": y_used, "idx": idx, "diag": diag}

    # Shared baseline conditions.
    add("clean_full", full_idx, ytr, None, "shared")
    add("noise_only_full", full_idx, y_noise_full, flip_full, "shared")
    add("imbalance_only_clean_sampling", idx_imb_clean, ytr, None, "shared", info_imb_clean)

    # Original process.
    add("compound_original_noise_then_observed_imbalance", idx_current, y_noise_full, flip_full, "original", info_current)

    # Clean-label sampling control / reversed process.
    add("compound_clean_label_imbalance_then_noise", idx_imb_clean, y_noise_after_clean_imb_full, flip_after_clean_imb_full, "clean_sampling_control", info_imb_clean)

    # Fixed current-index controls.
    add("imbalance_matched_current_indices_clean_labels", idx_current, ytr, None, "fixed_current_index", info_current)
    add("compound_fixed_current_indices_noisy_labels", idx_current, y_noise_full, flip_full, "fixed_current_index", info_current)

    # Equal training size controls.
    add("clean_equal_size", idx_clean_equal, ytr, None, "equal_size")
    add("noise_only_equal_size", idx_noise_equal, y_noise_equal_full, flip_equal_full, "equal_size")
    add("imbalance_only_equal_size", idx_imb_equal, ytr, None, "equal_size", info_imb_clean)
    add("compound_original_equal_size", idx_current_equal, y_noise_full, flip_full, "equal_size", info_current)

    return conditions


In [ ]:
# =========================
# MAIN RUNNER
# =========================


def run_v11_process_controls():
    out_dir = ensure_dir(Path(OUT_ROOT) / "process_controls")
    raw_path = out_dir / "process_control_condition_metrics.csv"
    failed_path = out_dir / "process_control_failed_runs.csv"

    completed = set()
    if RESUME and raw_path.exists():
        existing = pd.read_csv(raw_path)
        completed = set(existing["run_key"].astype(str))
        print("Resume mode. Completed rows:", len(completed))

    condition_names = [
        "clean_full",
        "noise_only_full",
        "imbalance_only_clean_sampling",
        "compound_original_noise_then_observed_imbalance",
        "compound_clean_label_imbalance_then_noise",
        "imbalance_matched_current_indices_clean_labels",
        "compound_fixed_current_indices_noisy_labels",
        "clean_equal_size",
        "noise_only_equal_size",
        "imbalance_only_equal_size",
        "compound_original_equal_size",
    ]

    for dataset_name in PMLB_DATASETS:
        print("\nDATASET:", dataset_name)
        try:
            X, y, meta = load_pmlb_dataset(dataset_name)
            valid, reason = validate_dataset(X, y)
            if not valid:
                save_row({"dataset_name": dataset_name, "stage": "validate", "error": reason}, failed_path)
                print("Skip", reason)
                continue
        except Exception as e:
            save_row({"dataset_name": dataset_name, "stage": "load", "error": str(e)[:500]}, failed_path)
            print("Load failed", e)
            continue

        for seed in SEEDS:
            try:
                Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=TEST_SIZE, random_state=seed, stratify=y)
                Xtr = Xtr.reset_index(drop=True)
                Xte = Xte.reset_index(drop=True)
                ytr = pd.Series(ytr).reset_index(drop=True)
                yte = pd.Series(yte).reset_index(drop=True)
                conditions = make_conditions_for_seed(Xtr, ytr, seed)
            except Exception as e:
                save_row({"dataset_name": dataset_name, "seed": seed, "stage": "split_or_conditions", "error": str(e)[:500]}, failed_path)
                print("Condition failed", seed, e)
                continue

            for model_name in MODELS:
                for condition_name in condition_names:
                    run_key = f"{dataset_name}__s{seed}__{model_name}__{condition_name}"
                    if run_key in completed:
                        continue
                    try:
                        c = conditions[condition_name]
                        if pd.Series(c["y"]).nunique() < 2:
                            raise ValueError("less_than_two_classes_in_training_condition")
                        row = fit_eval_condition(
                            dataset_name=dataset_name,
                            model_name=model_name,
                            seed=seed,
                            condition_name=condition_name,
                            Xtr=c["X"],
                            ytr_used=c["y"],
                            Xte=Xte,
                            yte=yte,
                            retained_idx=c["idx"],
                            extra_diag=c["diag"],
                        )
                        row.update({
                            "run_key": run_key,
                            "dataset_rows": int(len(X)),
                            "dataset_features": int(X.shape[1]),
                            "dataset_classes": int(pd.Series(y).nunique()),
                        })
                        save_row(row, raw_path)
                        completed.add(run_key)
                        if len(completed) % 250 == 0:
                            print("Completed condition rows:", len(completed))
                    except Exception as e:
                        save_row({
                            "run_key": run_key,
                            "dataset_name": dataset_name,
                            "model_name": model_name,
                            "seed": seed,
                            "condition_name": condition_name,
                            "stage": "fit_eval",
                            "error": str(e)[:500],
                        }, failed_path)
    print("Saved raw metrics:", raw_path)
    return raw_path, failed_path


In [ ]:
# =========================
# INTERACTION ANALYSIS
# =========================


def compute_process_control_interactions(raw_path):
    out_dir = ensure_dir(Path(OUT_ROOT) / "process_controls")
    df = pd.read_csv(raw_path)

    metric_cols = ["macro_f1", "balanced_accuracy", "accuracy", "mcc", "minority_recall"]
    variants = [
        {
            "process_variant": "original_noise_then_observed_imbalance",
            "clean": "clean_full",
            "noise": "noise_only_full",
            "imbalance": "imbalance_only_clean_sampling",
            "compound": "compound_original_noise_then_observed_imbalance",
            "interpretation": "main process: noise changes observed labels before imbalance sampling",
        },
        {
            "process_variant": "clean_label_imbalance_then_noise",
            "clean": "clean_full",
            "noise": "noise_only_full",
            "imbalance": "imbalance_only_clean_sampling",
            "compound": "compound_clean_label_imbalance_then_noise",
            "interpretation": "control: imbalance retained set is computed from clean labels before noise",
        },
        {
            "process_variant": "fixed_current_index_matched_baseline",
            "clean": "clean_full",
            "noise": "noise_only_full",
            "imbalance": "imbalance_matched_current_indices_clean_labels",
            "compound": "compound_fixed_current_indices_noisy_labels",
            "interpretation": "control: compound retained rows are reused for the matched imbalance baseline",
        },
        {
            "process_variant": "equal_training_size_control",
            "clean": "clean_equal_size",
            "noise": "noise_only_equal_size",
            "imbalance": "imbalance_only_equal_size",
            "compound": "compound_original_equal_size",
            "interpretation": "control: clean, single, and compound conditions have matched training size",
        },
    ]

    rows = []
    key_cols = ["dataset_name", "model_name", "seed"]

    for keys, g in df.groupby(key_cols):
        vals = {r["condition_name"]: r for _, r in g.iterrows()}
        for v in variants:
            needed = [v["clean"], v["noise"], v["imbalance"], v["compound"]]
            if not all(n in vals for n in needed):
                continue
            for metric in metric_cols:
                try:
                    p0 = float(vals[v["clean"]][metric])
                    pn = float(vals[v["noise"]][metric])
                    pi = float(vals[v["imbalance"]][metric])
                    pc = float(vals[v["compound"]][metric])
                    if any(pd.isna(x) for x in [p0, pn, pi, pc]):
                        continue
                    noise_drop = p0 - pn
                    imbalance_drop = p0 - pi
                    compound_drop = p0 - pc
                    interaction = compound_drop - (noise_drop + imbalance_drop)
                    rows.append({
                        "dataset_name": keys[0],
                        "model_name": keys[1],
                        "seed": keys[2],
                        "process_variant": v["process_variant"],
                        "metric": metric,
                        "clean_condition": v["clean"],
                        "noise_condition": v["noise"],
                        "imbalance_condition": v["imbalance"],
                        "compound_condition": v["compound"],
                        "p_clean": p0,
                        "p_noise": pn,
                        "p_imbalance": pi,
                        "p_compound": pc,
                        "noise_drop": noise_drop,
                        "imbalance_drop": imbalance_drop,
                        "observed_compound_drop": compound_drop,
                        "expected_additive_drop": noise_drop + imbalance_drop,
                        "interaction": interaction,
                        "interpretation": v["interpretation"],
                    })
                except Exception:
                    continue

    inter = pd.DataFrame(rows)
    inter_path = out_dir / "process_control_interactions.csv"
    inter.to_csv(inter_path, index=False)

    # Main summaries.
    summary = inter.groupby(["process_variant", "metric"])[[
        "noise_drop", "imbalance_drop", "observed_compound_drop", "expected_additive_drop", "interaction"
    ]].mean().reset_index()
    summary.to_csv(out_dir / "process_control_summary_all_metrics.csv", index=False)

    macro = summary[summary["metric"] == "macro_f1"].copy()
    macro.to_csv(out_dir / "process_control_summary_macro_f1.csv", index=False)

    # Dataset-equal summary for Macro-F1.
    macro_inter = inter[inter["metric"] == "macro_f1"].copy()
    ds_equal = macro_inter.groupby(["process_variant", "dataset_name"])["interaction"].mean().reset_index()
    ds_equal_summary = ds_equal.groupby("process_variant")["interaction"].agg(["mean", "std", "count"]).reset_index()
    ds_equal_summary.to_csv(out_dir / "process_control_dataset_equal_summary_macro_f1.csv", index=False)

    # Positive dataset percentages.
    pos_ds = ds_equal.assign(positive=lambda x: x["interaction"] > 0).groupby("process_variant")["positive"].mean().reset_index()
    pos_ds.rename(columns={"positive": "positive_dataset_fraction"}, inplace=True)
    pos_ds.to_csv(out_dir / "process_control_positive_dataset_fraction_macro_f1.csv", index=False)

    print("Saved interactions:", inter_path)
    print("Macro-F1 summary:")
    display(macro)
    print("Dataset-equal Macro-F1 interaction summary:")
    display(ds_equal_summary)
    print("Positive dataset fraction:")
    display(pos_ds)
    return inter, summary, macro


In [ ]:
# =========================
# COMPOSITION / RETAINED-INDEX SUMMARIES
# =========================


def summarize_composition(raw_path):
    out_dir = ensure_dir(Path(OUT_ROOT) / "process_controls")
    df = pd.read_csv(raw_path)

    # One row per dataset/seed/condition is enough because composition repeats across models.
    comp_cols = [
        "dataset_name", "seed", "condition_name", "process_family", "train_n",
        "clean_minority_retained_count", "clean_minority_retention_rate",
        "effective_clean_minority_ratio", "clean_true_majority_ratio_retained",
        "clean_true_min_class_count_retained", "mislabeled_retained_count",
        "mislabeled_retained_ratio", "mislabeled_clean_minority_count",
        "mislabeled_clean_majority_count"
    ]
    keep_cols = [c for c in comp_cols if c in df.columns]
    comp = df[keep_cols].drop_duplicates()
    comp.to_csv(out_dir / "process_control_composition_by_condition.csv", index=False)

    summary_cols = [
        "train_n", "clean_minority_retention_rate", "effective_clean_minority_ratio",
        "clean_true_majority_ratio_retained", "mislabeled_retained_ratio"
    ]
    available = [c for c in summary_cols if c in comp.columns]
    comp_summary = comp.groupby(["condition_name", "process_family"])[available].mean().reset_index()
    comp_summary.to_csv(out_dir / "process_control_composition_summary.csv", index=False)

    display(comp_summary)
    return comp, comp_summary


In [ ]:
# =========================
# WRITE REPRODUCIBILITY README
# =========================


def write_readme():
    repo = ensure_dir(Path(OUT_ROOT) / "reproducibility_notes")
    text = f"""# V11 Process-Control Experiments

This package contains the final process-control experiments for the compound-corruption benchmark.

## Purpose

The controls distinguish a process-dependent sampling-frame effect from a generic property of label noise and class imbalance.

## Fixed experimental setting

- datasets: {len(PMLB_DATASETS)} PMLB datasets
- models: {len(MODELS)} models
- seeds: {SEEDS}
- label noise: {NOISE_RATE}
- induced majority ratio: {IMBALANCE_RATIO}
- missingness: 0.0

## Process variants

1. `original_noise_then_observed_imbalance`: label noise is applied first, then imbalance sampling is computed from corrupted observed labels.
2. `clean_label_imbalance_then_noise`: imbalance sampling is computed from clean labels, then retained labels are corrupted.
3. `fixed_current_index_matched_baseline`: the original compound retained rows are reused to construct a matched clean-label imbalance baseline.
4. `equal_training_size_control`: clean, single-corruption, and compound conditions are compared after matching training-set size.

## Main output files

- `process_control_condition_metrics.csv`: run-level condition metrics.
- `process_control_interactions.csv`: additive-null process-control interactions.
- `process_control_summary_macro_f1.csv`: compact Macro-F1 summary.
- `process_control_dataset_equal_summary_macro_f1.csv`: dataset-equal Macro-F1 summary.
- `process_control_composition_summary.csv`: composition and retained-evidence diagnostics.

## Interpretation guidance

If the original process has a much larger interaction than the clean-label or fixed-index controls, the correct conclusion is not that the original effect is invalid. The correct conclusion is that the large excess degradation is specifically tied to the data-construction process in which corrupted observed labels define the later imbalance sampling frame.
"""
    (repo / "README_v11_process_controls.md").write_text(text, encoding="utf-8")
    return repo


In [ ]:
# =========================
# EXECUTE ALL
# =========================

start = time.time()

print("STEP 1: Dataset manifest")
manifest, failed_manifest = save_manifest()

print("\nSTEP 2: Process-control runs")
raw_path, failed_path = run_v11_process_controls()

print("\nSTEP 3: Interaction analysis")
inter, summary, macro_summary = compute_process_control_interactions(raw_path)

print("\nSTEP 4: Composition summaries")
comp, comp_summary = summarize_composition(raw_path)

print("\nSTEP 5: README")
write_readme()

print("\nSTEP 6: Zip outputs")
zip_base = "/kaggle/working/v11_process_control_outputs"
zip_path = zip_base + ".zip"
if Path(zip_path).exists():
    Path(zip_path).unlink()
shutil.make_archive(zip_base, "zip", OUT_ROOT)

print("Final ZIP:", zip_path)
print("Runtime hours:", round((time.time() - start) / 3600, 3))
